#📝 Evidencia de aprendizaje (EA2). Procesamiento de datos en una infraestructura cloud con Databricks (Free Edition)

**Integrante:** Aida Luz Aguirre Hernandez

**Curso:** Big Data

**Grupo:** grupo_8


## Introducción
Este trabajo forma parte del proceso de aprendizaje sobre el análisis y procesamiento de datos utilizando plataformas modernas en la nube. En esta Actividad 2 se construye un flujo completo de ingestión, modelado, consulta y validación de un dataset dentro de Databricks Free Edition, la versión actual disponible para entornos educativos y de práctica.

Este ejercicio se desarrolla utilizando un dataset real de comercio electrónico obtenido desde Kaggle, lo cual permite simular un escenario típico de análisis de datos: diseño del esquema, carga en un entorno distribuido, creación de tablas administradas y consultas tanto en SQL como en PySpark.

## Objetivo general
Implementar un flujo funcional de procesamiento de datos en Databricks Free Edition, que incluya diseño del esquema, ingestión del dataset, creación de tablas, ejecución de validaciones y comparación entre consultas SQL y PySpark.

## Objetivos específicos
- Diseñar el esquema de almacenamiento para el dataset seleccionado, definiendo entidades, atributos, tipos de datos y relaciones.
- Configurar y documentar el entorno de trabajo en Databricks Free Edition (clúster, runtime, DBFS y configuración interna).
- Ingerir el dataset desde Kaggle hacia el almacenamiento de Databricks y cargarlo en Spark aplicando el esquema definido.
- Crear una tabla administrada o externa y validar su correcta creación mediante consultas SQL y operaciones PySpark.
- Realizar consultas exploratorias, agregaciones y análisis estadístico para validar la integridad de los datos.
- Comparar ventajas y desventajas del uso de SQL frente a PySpark en un entorno de procesamiento distribuido.

## Dataset utilizado
El dataset utilizado proviene de Kaggle: [An Online Shop Business](https://www.kaggle.com/datasets/gabrielramos87/an-online-shop-business), el cual contiene información transaccional de ventas en un comercio electrónico: identificadores de productos, clientes, países, cantidades y fechas.

Este conjunto de datos es ideal para practicar técnicas de ingestion, modelado y análisis dentro de un entorno distribuido como Databricks.



# 1. Diseño del esquema del dataset

A partir del dataset [An Online Shop Business](https://www.kaggle.com/datasets/gabrielramos87/an-online-shop-business), se propone un modelo basado en tres entidades principales: **CUSTOMER**(clientes), **PRODUCT**(productos) y **SALES**(ventas).  

### 1.1 Descripción de entidades y campos clave del dataset

El dataset An Online Shop Business contiene información transaccional de un negocio de comercio electrónico. A partir de la estructura observada en los datos, se identifican tres entidades principales. A continuación se describen sus campos, tipos de datos, llaves y nulabilidad.



**🟦 Entidad  de dimensión: CUSTOMER**
Representa a los clientes registrados en las transacciones.

| Campo en Español    | Campo en inglés       | Tipo   | Llave | Nulabilidad | Descripción |
|---------------------|----------------|--------|-------|-------------|-------------|
| ID del cliente      | customer_id    | STRING  | PK    | NOT NULL    | Identificador del cliente.|
| País                | country        | STRING | -     | NOT NULL    | País asociado a la compra. |


**🟦 Entidad de dimensión: PRODUCT**
Agrupa la información básica de los productos.

| Campo en Español       | Campo en inglés        | Tipo   | Llave | Nulabilidad | Descripción |
|------------------------|---------------|--------|-------|-------------|-------------|
| ID del producto        | product_id    | STRING | PK    | NOT NULL    | Identificador del producto. |
| Nombre del producto    | product_name  | STRING | -     | NOT NULL    | Nombre del producto. |


**🟦 Entidad de Hechos: SALES**
Registra cada línea de venta en el dataset.

| Campo en Español         | Campo en inglés           | Tipo   | Llave | Nulabilidad | Descripción |
|--------------------------|-------------------|--------|-------|-------------|-------------|
| ID de la transacción     | transaction_id    | STRING | PK    | NOT NULL    | Identificador de la transacción. |
| Fecha                    | date              | DATE   | -     | NOT NULL    | Fecha en la que se realizó la compra. |
| ID del cliente           | customer_id       | STRING  | FK    | NOT NULL    | Referencia al cliente.|
| ID del producto          | product_id        | STRING | FK    | NOT NULL    | Referencia al producto. |
| Cantidad                 | quantity          | INT    | -     | NOT NULL    | Unidades vendidas. |
| Precio unitario          | Price            | FLOAT  | -     | NOT NULL    | Precio por unidad del producto en el momento de la compra. |
| Pago total             | TotalPrice       | FLOAT  | -     | NOT NULL    | Valor total de la venta (Price * Quantity). |


### 1.2 DDL Spark SQL  

Se agrega el archivo .sql que contiene las sentencias DDL utilizadas para construir el esquema presentado. [Archivo DDL.sql](Archivo DDL.sql)

### 1.3  diagrama simple

![ERD TechStore Online SA](/files/DiagramaSimple.png)


## 2. Configuración de Databricks 
 En Databricks Free Edition lo siguiente no es posible, así que solo se muestran los parámetros esperados

**2.1 Crear y configurar el clúster**

1. En la barra lateral a la izquierda, clic en **Compute** o **Clusters**.
2. Da clic en **Create Cluster**.
3. Configura los campos recomendados por ejemplo:
   - **Cluster Name:** cluster_online_shop
   - **Databricks Runtime Version:** elegir la versión disponible más reciente o LTS en tu workspace 13.0 LTS 
   - **Python Version:** Python version: 3.12.3.
   - **Cluster Mode:** Standard
   - **Autoscaling:** Activado (Min 1 – Max 4)
   - **Worker Type / Worker size:** 4 vCPU / 16 GB RAM
   - **Driver/Worker cores & memory:** según la UI; en CE suelen ser fijos y menores.
4. Clic en **Create cluster**. 

La configuración anterior generaría un clúster escalable para ejecutar Spark y SQL.

Después de crear el clúter asi se vera la configuración realizada

- Nombre del clúster: cluster_online_shop
- Databricks Runtime: 13.0 LTS
- Python: 3.12.3
- Modo: Standard
- Núcleos/RAM: 4 vCPU / 16 GB RAM
- Autoscaling: 1– 4 nodos

**2.2 Versiones de Python y Spark**

El comando:
for item in spark.sparkContext.getConf().getAll(): 
print(item)

Este comando muestra la versión de Spark, la configuración asociada a Python, así como los directorios utilizados y los recursos asignados (núcleos y memoria).

Debido a las limitaciones de Databricks Free Edition, esta información no puede recuperarse totalmente, por lo que únicamente se reportan las versiones de Spark y Python.


In [0]:
# Celda Python - mostrar versiones
import sys
print("Versión de Spark:", spark.version)
print("Versión de Python:", sys.version.splitlines()[0])


Versión de Spark: 4.0.0
Versión de Python: 3.12.3 (main, Aug 14 2025, 17:47:21) [GCC 13.3.0]


In [0]:
import sys
# Versión de Spark
print("Python version:", sys.version)

# versión de Python
print("\nSpark version:", spark.version)


Python version: 3.12.3 (main, Aug 14 2025, 17:47:21) [GCC 13.3.0]

Spark version: 4.0.0


**2.3 Estructura de almacenamiento**

el almacenamiento se gestiona mediante DBFS, usando la ruta /FileStore/datasets/ para cargar el dataset proveniente de kaggle.

### 3. Obtención del dataset


**3.1** Instalación de librerías y descarga del dataset

 




In [0]:
%pip install kagglehub pandas




  Using cached kagglehub-0.3.13-py3-none-any.whl.metadata (38 kB)
Using cached kagglehub-0.3.13-py3-none-any.whl (68 kB)
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


A continuación se realiza la Descarga, Extracción y el Lectura del Dataset desde Kaggle.

In [0]:
# Importación librerias
import os
import zipfile
import kagglehub
import pandas as pd

# Descargar, Extraer y el Leer el Dataset
def download_dataset_zip(url=""):
    print("Descargando dataset desde Kaggle...")
    dataset_path = kagglehub.dataset_download(url)
    print("Ruta al dataset:", dataset_path)
    return dataset_path


def extract_zip_files(dataset_path):
    zip_files = [f for f in os.listdir(dataset_path) if f.endswith('.zip')]
    
    # Si hay ZIP → extraer
    if zip_files:
        zip_file = os.path.join(dataset_path, zip_files[0])
        extract_dir = os.path.join(dataset_path, "extracted")
        os.makedirs(extract_dir, exist_ok=True)

        print(f"Extrayendo {zip_file} en {extract_dir}...")
        with zipfile.ZipFile(zip_file, "r") as z:
            z.extractall(extract_dir)
        return extract_dir
    
    # Si no hay ZIP pero sí CSV → dataset ya extraído
    csv_files = [f for f in os.listdir(dataset_path) if f.endswith('.csv')]
    if csv_files:
        print("No se encontró archivo ZIP, pero sí archivos CSV. Se asume que ya está extraído.")
        return dataset_path
    
    raise FileNotFoundError("No se encontró ningún archivo ZIP ni CSV.")


def create_csv(csv_dir, csv_name=None):
    if csv_name:  # archivo especificado
        file_path = os.path.join(csv_dir, csv_name)
        print(f"Leyendo {file_path}...")
        df = pd.read_csv(file_path)
        print("CSV cargado correctamente.")
        return df
    
    # detectar automáticamente
    csv_files = [f for f in os.listdir(csv_dir) if f.endswith('.csv')]
    if not csv_files:
        raise FileNotFoundError("No se encontraron archivos CSV en el directorio extraído.")
    
    file_path = os.path.join(csv_dir, csv_files[0])
    print(f"Leyendo {file_path}...")
    df = pd.read_csv(file_path)
    print("CSV cargado correctamente.")
    return df


 Descarga del dataset

In [0]:
df = pd.DataFrame()
dataset_path = download_dataset_zip("gabrielramos87/an-online-shop-business")
csv_dir = extract_zip_files(dataset_path)
df = create_csv( csv_dir, csv_name="Sales Transaction v.4a.csv")


Descargando dataset desde Kaggle...


100%|██████████| 6.66M/6.66M [00:00<00:00, 11.7MB/s]

Extracting files...


Ruta al dataset: /home/spark-344fc438-294d-4227-981c-06/.cache/kagglehub/datasets/gabrielramos87/an-online-shop-business/versions/7
No se encontró archivo ZIP, pero sí archivos CSV. Se asume que ya está extraído.
Leyendo /home/spark-344fc438-294d-4227-981c-06/.cache/kagglehub/datasets/gabrielramos87/an-online-shop-business/versions/7/Sales Transaction v.4a.csv...
CSV cargado correctamente.


In [0]:
df.head()



,TransactionNo,Date,ProductNo,ProductName,Price,Quantity,CustomerNo,Country
0,581482,12/9/2019,22485,Set Of 2 Wooden Market Crates,21.47,12,17490.0,United Kingdom
1,581475,12/9/2019,22596,Christmas Star Wish List Chalkboard,10.65,36,13069.0,United Kingdom
2,581475,12/9/2019,23235,Storage Tin Vintage Leaf,11.53,12,13069.0,United Kingdom
3,581475,12/9/2019,23272,Tree T-Light Holder Willie Winkie,10.65,12,13069.0,United Kingdom
4,581475,12/9/2019,23239,Set Of 4 Knick Knack Tins Poppies,11.94,6,13069.0,United Kingdom


In [0]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 536350 entries, 0 to 536349
Data columns (total 8 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   TransactionNo  536350 non-null  object 
 1   Date           536350 non-null  object 
 2   ProductNo      536350 non-null  object 
 3   ProductName    536350 non-null  object 
 4   Price          536350 non-null  float64
 5   Quantity       536350 non-null  int64  
 6   CustomerNo     536295 non-null  float64
 7   Country        536350 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 32.7+ MB


**Preparación del catálogo y esquema para las tablas del proyecto**

In [0]:
%sql
-- 1. Crear el catálogo principal si aún no existe
-- Este catálogo agrupa todas las bases de datos relacionadas con el proyecto de tienda online
CREATE CATALOG IF NOT EXISTS online_shop;

-- 2. Crear el esquema (base de datos) para las tablas de ventas y dimensiones
-- Aquí se almacenarán las tablas de hechos y dimensiones del dataset
CREATE SCHEMA IF NOT EXISTS online_shop.sales_schema;

-- 3. Crear un volumen para almacenar archivos no tabulares o CSVs
-- Este volumen servirá para guardar datasets crudos antes de procesarlos
CREATE VOLUME IF NOT EXISTS online_shop.sales_schema.vol_sales_data;


**Convertir el DataFrame de pandas a un DataFrame de Spark**

In [0]:
# permite trabajar con Spark SQL y realizar operaciones distribuidas
spark_df = spark.createDataFrame(df)

# Mostrar las primeras filas para verificar que la conversión fue exitosa
spark_df.show(5)


+-------------+---------+---------+--------------------+-----+--------+----------+--------------+
|TransactionNo|     Date|ProductNo|         ProductName|Price|Quantity|CustomerNo|       Country|
+-------------+---------+---------+--------------------+-----+--------+----------+--------------+
|       581482|12/9/2019|    22485|Set Of 2 Wooden M...|21.47|      12|   17490.0|United Kingdom|
|       581475|12/9/2019|    22596|Christmas Star Wi...|10.65|      36|   13069.0|United Kingdom|
|       581475|12/9/2019|    23235|Storage Tin Vinta...|11.53|      12|   13069.0|United Kingdom|
|       581475|12/9/2019|    23272|Tree T-Light Hold...|10.65|      12|   13069.0|United Kingdom|
|       581475|12/9/2019|    23239|Set Of 4 Knick Kn...|11.94|       6|   13069.0|United Kingdom|
+-------------+---------+---------+--------------------+-----+--------+----------+--------------+
only showing top 5 rows


**3.2 Creación de la tabla en Spark para el dataset de Online Shop Business**

- Se toma el DataFrame limpio spark_df que contiene los datos del negocio online.

- Los datos se guardan en formato Delta Lake, lo que permite manejar grandes volúmenes de información de manera eficiente y con soporte de transacciones.

- La tabla se registra en el Catálogo y Esquema de Databricks bajo el nombre tbl_venta_spk, quedando disponible para consultas con Spark SQL y análisis posteriores.

In [0]:
# Guarda el DataFrame de Spark como tabla Delta y registrarlo en el catálogo
spark_df.write.mode("overwrite").saveAsTable("online_shop.sales_schema.tbl_venta_spk")


**Descripción de la tabla creada en Spark**

- Se utiliza el comando DESCRIBE TABLE para ver la estructura de la tabla tbl_venta_spk registrada en Spark.
- Esto muestra los nombres de columnas, tipos de datos y otras propiedades, permitiendo verificar que los datos se cargaron correctamente.

In [0]:
%sql
DESCRIBE TABLE online_shop.sales_schema.tbl_venta_spk;


col_name,data_type,comment
TransactionNo,string,null
Date,string,null
ProductNo,string,null
ProductName,string,null
Price,double,null
Quantity,bigint,null
CustomerNo,double,null
Country,string,null


**Revisión de los datos de la tabla creada en Spark**

In [0]:
# 1. Cargamos la tabla Delta creada en Spark en un DataFrame
df_transacciones = spark.table("online_shop.sales_schema.tbl_venta_spk")

# 2. Seleccionamos únicamente las columnas numéricas (double, int, long)
#    para enfocarnos en los campos que permiten análisis estadístico.
#    Se omiten las columnas de tipo string o timestamp.
numeric_cols = [
    f.name for f in df_transacciones.schema
    if f.dataType.typeName() in ('double', 'decimal', 'float', 'integer', 'long')
]

# 3. Creamos un DataFrame solo con las columnas numéricas
#    y aplicamos describe() para obtener estadísticas básicas
df_numeric_stats = df_transacciones.select(*numeric_cols)

# Mostramos los resultados de manera más amigable con display()
display(df_numeric_stats.describe())



summary,Price,Quantity,CustomerNo
count,536350,536350,536295
mean,12.662182287685862,9.919347441036637,15227.893178194838
stddev,8.490450200816934,216.66229978945705,1716.5829320559167
min,5.13,-80995,12004.0
max,660.62,80995,18287.0


**Verificación de las tablas creadas en el esquema de sales_schema**

In [0]:
%sql
SHOW TABLES IN online_shop.sales_schema;


database,tableName,isTemporary
sales_schema,tbl_venta_spk,false


**Creación de tabla con SQL y verificación del dataset**

In [0]:
%sql
-- Listar los archivos dentro del directorio del Volume
LIST 'dbfs:/Volumes/online_shop/default/vol_ventas_trasaccion/';


path,name,size,modification_time
dbfs:/Volumes/online_shop/default/vol_ventas_trasaccion/Sales Transaction v.4a.csv,Sales Transaction v.4a.csv,42995360,1763784348000


**3.3 Carga de datos en Spark desde el volumen**

In [0]:
# 1. Definir la ruta del CSV dentro del volumen de Unity Catalog o DBFS
#    Aquí se encuentra el dataset de Online Shop Business que vamos a analizar
ruta_csv_volume = 'dbfs:/Volumes/online_shop/default/vol_ventas_trasaccion/'

# 2. Leer el archivo CSV directamente en un DataFrame de Spark
#    No se crea todavía una tabla Delta, solo se carga la información para inspección
df_diagnostico = spark.read.csv(
    ruta_csv_volume,
    header=True,      # La primera fila contiene los nombres de las columnas
    inferSchema=True  # Spark detecta automáticamente los tipos de datos
)

# 3. Mostrar los nombres de las columnas que Spark detectó
#    Esto nos ayuda a verificar que la lectura fue correcta
print(df_diagnostico.columns)


['TransactionNo', 'Date', 'ProductNo', 'ProductName', 'Price', 'Quantity', 'CustomerNo', 'Country']


**Descripción de la tabla en una vista temporal** 

Creamos una vista temporal a partir del archivo CSV ubicado en DBFS. Esta vista permite que Spark lea los datos sin necesidad de crear todavía una tabla permanente. Luego usamos DESCRIBE para inspeccionar los nombres de columnas y tipos de datos detectados automáticamente, validando que la lectura del CSV se haya realizado correctamente.

In [0]:
%sql
-- Generar una vista temporal a partir del archivo CSV ubicado en la ruta donde esta ubicado el volumen
CREATE OR REPLACE TEMPORARY VIEW raw_csv_view 
USING CSV
OPTIONS (
  'path' = 'dbfs:/Volumes/online_shop/default/vol_ventas_trasaccion/',
  'header' = 'true',
  'inferSchema' = 'true',
  'timestampFormat' = 'yyyy-MM-dd HH:mm:ss'
);

--  Verificación: inspeccionar las columnas y tipos que Spark detectó automáticamente
DESCRIBE raw_csv_view;


col_name,data_type,comment
TransactionNo,string,null
Date,date,null
ProductNo,string,null
ProductName,string,null
Price,double,null
Quantity,int,null
CustomerNo,string,null
Country,string,null


Creamos la tabla gestionada 'tbl_venta' en el esquema 'sales_schema' y la poblamos con los datos de la vista temporal 'raw_csv_view'. Esto permite almacenar permanentemente los datos del CSV en Spark, manteniendo las columnas y tipos detectados, para poder consultarlos y analizarlos posteriormente.

In [0]:
%sql
-- Crear la tabla poblandola con la vista temporal
CREATE TABLE IF NOT EXISTS online_shop.sales_schema.tbl_venta
AS SELECT
    TransactionNo,
    Date,
    ProductNo,
    ProductName,
    Price,
    Quantity,
    CustomerNo,
    Country
FROM raw_csv_view;


num_affected_rows,num_inserted_rows


Contamos los registros totales

In [0]:
%sql
SELECT COUNT(*) AS total_registros
FROM online_shop.sales_schema.tbl_venta;

total_registros
536350


### 4 Validaciones en Spark y SQL

**4.1 Metadatos**

Descripción de la tabla Delta permanente

In [0]:
%sql
-- Muestra la estructura de la tabla tbl_venta registrada permanentemente
DESCRIBE TABLE online_shop.sales_schema.tbl_venta;



col_name,data_type,comment
TransactionNo,string,null
Date,date,null
ProductNo,string,null
ProductName,string,null
Price,double,null
Quantity,int,null
CustomerNo,string,null
Country,string,null


Verificación de existencia y registro en el catálogo

In [0]:
%sql

SHOW CREATE TABLE online_shop.sales_schema.tbl_venta;



createtab_stmt
"CREATE TABLE online_shop.sales_schema.tbl_venta ( TransactionNo STRING, Date DATE, ProductNo STRING, ProductName STRING, Price DOUBLE, Quantity INT, CustomerNo STRING, Country STRING) USING delta COLLATION 'UTF8_BINARY' TBLPROPERTIES ( 'delta.enableDeletionVectors' = 'true', 'delta.feature.appendOnly' = 'supported', 'delta.feature.deletionVectors' = 'supported', 'delta.feature.invariants' = 'supported', 'delta.minReaderVersion' = '3', 'delta.minWriterVersion' = '7', 'delta.parquet.compression.codec' = 'zstd')"


Inspección del esquema en Spark

In [0]:
# Esto permite ver los nombres de las columnas, los tipos de datos inferidos y si permiten valores nulos
spark_df = spark.table("online_shop.sales_schema.tbl_venta")
spark_df.printSchema()


root
 |-- TransactionNo: string (nullable = true)
 |-- Date: date (nullable = true)
 |-- ProductNo: string (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- CustomerNo: string (nullable = true)
 |-- Country: string (nullable = true)



**4.2 Descripción de datos**

- Descripción estadística de las columnas numéricas del dataset de transacciones 

In [0]:
# Cargamos la tabla permanente a un DataFrame de Spark
df_transacciones = spark.table("online_shop.sales_schema.tbl_venta")

# Identificamos las columnas de tipo numérico (double, int, long)
#    Excluimos columnas de tipo string o date.
numeric_cols = [
    f.name for f in df_transacciones.schema
    if f.dataType.typeName() in ('double', 'decimal', 'float', 'integer', 'long')
]

# Creamos un nuevo DataFrame con solo las columnas numéricas y aplicamos describe()
df_numeric_stats = df_transacciones.select(*numeric_cols)

# Mostramos estadísticas básicas (count, mean, stddev, min, max)
display(df_numeric_stats.describe())




summary,Price,Quantity
count,536350,536350
mean,12.662182287696377,9.919347441036637
stddev,8.490450200816914,216.6622997894527
min,5.13,-80995
max,660.62,80995


- Resumen estadístico de las columnas numéricas de la tabla de transacciones con SQL

In [0]:
%sql
SELECT
    'count' AS summary, 
    COUNT(Price) AS Price, 
    COUNT(Quantity) AS Quantity
FROM online_shop.sales_schema.tbl_venta

UNION ALL

SELECT 
    'mean' AS summary,
    ROUND(AVG(Price), 4),
    ROUND(AVG(Quantity), 4)
FROM online_shop.sales_schema.tbl_venta

UNION ALL

SELECT 
    'stddev' AS summary,
    ROUND(STDDEV(Price), 4),
    ROUND(STDDEV(Quantity), 4)
FROM online_shop.sales_schema.tbl_venta

UNION ALL

SELECT 
    'min' AS summary,
    MIN(Price),
    MIN(Quantity)
FROM online_shop.sales_schema.tbl_venta

UNION ALL

SELECT 
    'max' AS summary,
    MAX(Price),
    MAX(Quantity)
FROM online_shop.sales_schema.tbl_venta;


summary,Price,Quantity
mean,12.6622,9.9193
stddev,8.4905,216.6623
count,536350.0,536350.0
min,5.13,-80995.0
max,660.62,80995.0


**4.3Consultas SELECT y GROUP BY**

Total de ventas por país

- consutlta con SQL

In [0]:
%sql
SELECT
    Country,
    SUM(Price * Quantity) AS TotalSales,
    COUNT(*) AS NumTransactions
FROM online_shop.sales_schema.tbl_venta
GROUP BY Country
ORDER BY TotalSales DESC;


Country,TotalSales,NumTransactions
United Kingdom,4.9994030169984385E7,485095
Netherlands,2147811.390000003,2330
EIRE,1660645.0700000064,8048
Germany,1350265.4000000132,10675
France,1316880.9800000028,10526
Australia,988756.3500000024,1704
Sweden,396042.60999999975,417
Switzerland,358423.60999999923,2336
Japan,283293.46999999986,371
Belgium,271346.97999999975,2539


consulta con Spark

In [0]:
from pyspark.sql import functions as F

# Cargar la tabla en un DataFrame
df = spark.table("online_shop.sales_schema.tbl_venta")

# Agrupar por país y calcular totales
df_grouped = df.groupBy("Country") \
    .agg(
        F.sum(F.col("Price") * F.col("Quantity")).alias("TotalSales"),
        F.count("*").alias("NumTransactions")
    ) \
    .orderBy(F.desc("TotalSales"))

df_grouped.show() 

+---------------+--------------------+---------------+
|        Country|          TotalSales|NumTransactions|
+---------------+--------------------+---------------+
| United Kingdom|4.9994030169984385E7|         485095|
|    Netherlands|   2147811.390000003|           2330|
|           EIRE|  1660645.0700000064|           8048|
|        Germany|  1350265.4000000132|          10675|
|         France|  1316880.9800000028|          10526|
|      Australia|   988756.3500000024|           1704|
|         Sweden|  396042.60999999975|            417|
|    Switzerland|  358423.60999999923|           2336|
|          Japan|  283293.46999999986|            371|
|        Belgium|  271346.97999999975|           2539|
|          Spain|    265738.799999998|           2430|
|         Norway|   187544.7899999995|            938|
|       Portugal|  175269.44999999955|           1848|
|        Finland|  120597.86000000015|            692|
|        Denmark|  100439.12000000005|            416|
|Channel I

Ventas promedio por producto

- consulta con SQL

In [0]:
%sql
SELECT
    ProductName,
    AVG(Price * Quantity) AS AvgSales
FROM online_shop.sales_schema.tbl_venta
GROUP BY ProductName
ORDER BY AvgSales DESC
LIMIT 10;


ProductName,AvgSales
Paper Craft Little Birdie,250679.52500000002
Tea Time Tea Towels,16419.0
Small Chinese Style Scissor,2565.9414814814813
Asstd Design 3d Paper Stickers,2339.7536842105246
Mini Highlighter Pens,2177.7
Essential Balm 35g Tin In Envelope,1958.2867741935486
Potting Shed Candle Citronella,1816.9399999999996
Empire Design Rosette,1627.039655172414
Assorted Incense Pack,1551.0666666666668
Popart Wooden Pencils Asst,1297.5942028985507


- consulta con Spark

In [0]:
df_avg_sales = df.groupBy("ProductName") \
    .agg(F.avg(F.col("Price") * F.col("Quantity")).alias("AvgSales")) \
    .orderBy(F.desc("AvgSales")) \
    .limit(10)

df_avg_sales.show()


+--------------------+------------------+
|         ProductName|          AvgSales|
+--------------------+------------------+
|Paper Craft Littl...|250679.52500000002|
| Tea Time Tea Towels|           16419.0|
|Small Chinese Sty...|2565.9414814814813|
|Asstd Design 3d P...|2339.7536842105246|
|Mini Highlighter ...|            2177.7|
|Essential Balm 35...|1958.2867741935486|
|Potting Shed Cand...|1816.9399999999996|
|Empire Design Ros...| 1627.039655172414|
|Assorted Incense ...|1551.0666666666668|
|Popart Wooden Pen...|1297.5942028985507|
+--------------------+------------------+



Conteo de transacciones por fecha

- consulta con SQL

In [0]:
%sql
SELECT
    Date,
    COUNT(*) AS NumTransactions
FROM online_shop.sales_schema.tbl_venta
GROUP BY Date
ORDER BY Date;


Date,NumTransactions
2018-12-01,3086
2018-12-02,2101
2018-12-03,2150
2018-12-05,2709
2018-12-06,3851
2018-12-07,2926
2018-12-08,2629
2018-12-09,2826
2018-12-10,2740
2018-12-12,1450


- consulta con Spark

In [0]:
df_count_by_date = df.groupBy("Date") \
    .agg(F.count("*").alias("NumTransactions")) \
    .orderBy("Date")

df_count_by_date.show()


+----------+---------------+
|      Date|NumTransactions|
+----------+---------------+
|2018-12-01|           3086|
|2018-12-02|           2101|
|2018-12-03|           2150|
|2018-12-05|           2709|
|2018-12-06|           3851|
|2018-12-07|           2926|
|2018-12-08|           2629|
|2018-12-09|           2826|
|2018-12-10|           2740|
|2018-12-12|           1450|
|2018-12-13|           2273|
|2018-12-14|           2074|
|2018-12-15|           1341|
|2018-12-16|           1785|
|2018-12-17|           3087|
|2018-12-19|            520|
|2018-12-20|           1741|
|2018-12-21|           1561|
|2018-12-22|            289|
|2018-12-23|            955|
+----------+---------------+
only showing top 20 rows


### 5 Ventajas y desventajas: SQL vs Spark


peGAR IMAGEN  VENTAJAS Y DESVENTAJAS
